In [1]:
import torch
import torch.nn.functional as F
import json
import sys
import numpy as np
import textwrap

In [2]:
# ── Environment setup ─────────────────────────────────────────────────────
# Run this cell first every Colab session (or once on a local machine).
# It mounts Drive (Colab only) and adds the repo root to sys.path so all
# project imports work correctly.
import os, sys

if os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

# Add the repo root to the module search path
sys.path.insert(0, os.path.abspath(".."))   # works locally from notebooks/
                                             # adjust to REPO_ROOT on Colab

import config as cfg
cfg.make_dirs()
cfg.print_config()

Mounted at /content/drive


In [3]:
from model import GPT, GPTConfig, GPTWithKVCache
config = GPTConfig()
device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print("device:", device)

cpu


In [6]:
model

GPT(
  (token_emb): Embedding(8192, 512)
  (pos_emb): Embedding(256, 512)
  (blocks): ModuleList(
    (0-7): 8 x Block(
      (ln1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (c_attn): Linear(in_features=512, out_features=1536, bias=True)
        (c_proj): Linear(in_features=512, out_features=512, bias=True)
      )
      (ln2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (net): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=2048, out_features=512, bias=True)
        )
      )
    )
  )
  (ln_f): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=512, out_features=8192, bias=True)
)

In [6]:

if device == "cpu":
  ckpt = torch.load(cfg.PRETRAIN_FINAL_CKPT, map_location="cpu")
else:
  ckpt = torch.load(cfg.PRETRAIN_FINAL_CKPT)

model.load_state_dict(ckpt)

model.eval()
print("Model loaded successfully")

Model loaded successfully


In [9]:
if device == "cpu":
  torch.set_num_threads(4)

In [7]:
from tokenizers import ByteLevelBPETokenizer

tokenizer = ByteLevelBPETokenizer(
    cfg.TOKENIZER_VOCAB,
    cfg.TOKENIZER_MERGES
)

In [8]:
prompt = "Once upon a time"

input_ids = tokenizer.encode(prompt).ids
x = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)

In [16]:
from generation.sampler import generate, encode_prompt

In [11]:
# output = generate_basic(
#     model,
#     x,
#     max_new_tokens=100,
#     temperature=0.8,
#     top_k=50
# )

with torch.no_grad():
  output = generate(
      model,
      x,
      max_new_tokens=120,
      temperature=0.9,
      top_k=50,
      top_p=0.9,
      repetition_penalty=1.15
  )

In [12]:
tokens = output[0].tolist()

text = tokenizer.decode(tokens)

print(textwrap.fill(text, 80))

Once upon a time, there was a little girl named Lily. She loved to draw pictures
with her chalk on paper. One day, she drew a picture of her family. It was an
ancient art box because she had used it for years.  Suddenly, Lily's drawing
fell out of ink and landed in the mailbox. She felt sad and worried. But then,
her mom came outside and saw what happened. "Don't worry, we can clean your
drawings," said Lily.   They went inside and found some tape, and made a big
picture. Then, they put the picture back together and made it look


In [17]:
%%time
# Generate Multiple Samples
for _ in range(5):

    output = generate(
        model,
        x,
        max_new_tokens=200,
        temperature=0.9,
        top_k=50
    )
    generated_txt = tokenizer.decode(output[0].tolist())
    print(textwrap.fill(generated_txt, 80))
    print("\n---\n")

Once upon a time there was a brave little dog. He always wanted to do nice
things, even when it was difficult to do.  One day he went to the park and saw
something exciting. The ground beneath his feet were big and he decided to jump
through it. He started to slide down as fast as he could but it made him scared.
So he quickly grabbed hold of Sam's leash and pushed down the hill once more.
But it was too late and he never made it. He slipped and fell in many ways.
Just when he finally reached the bottom of the hill he had gone away on top. He
had to be very quiet. But then he realized that he was safe and still.  With
that, he went home feeling happy and proud. His adventure was over!Once upon a
time, there was a little girl named Lucy. She liked to make a cake. Every day,
she would mix two dessers with butter and put them around her finger. She
thought it was incredible

---

Once upon a time, there was a little birdie who loved to fly. He would flap his
wings and see all the pretty t